# QAOA for triangle MaxCut

Compare the p=1 QAOA cost landscape for a three-node triangle graph.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
cost = SparsePauliOp.from_list([
    ("III", 1.5), ("IZZ", -0.5), ("ZZI", -0.5), ("ZIZ", -0.5)
])
parameters = [(gamma, beta) for gamma in np.linspace(0.0, np.pi, 7) for beta in np.linspace(0.0, np.pi / 2, 5)]

def qaoa_circuit(gamma, beta):
    circuit = QuantumCircuit(3)
    circuit.h(range(3))
    for first, second in ((0, 1), (1, 2), (0, 2)):
        circuit.rzz(float(-gamma), first, second)
    for wire in range(3):
        circuit.rx(float(2 * beta), wire)
    return circuit

circuits = [qaoa_circuit(*values) for values in parameters]

def reference_costs():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, cost)]).result()[0].data.evs.item() for c in circuits])

reference, reference_ms, _ = benchmark(reference_costs)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_costs():
    return np.asarray([estimator.run([(c, cost)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_costs)
error = max_abs_error(reference, candidate)
best_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = qiskit_selection(estimator)
tutorial_result = emit_result(
    notebook="qiskit/11_qaoa_maxcut.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="QAOA cost landscape atol=3e-6",
    passed=error <= 3e-6 and best_match,
    exact_match=best_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_cost_error": error, "reference_best": float(reference.max()), "mettleq_best": float(candidate.max()), "best_parameters": parameters[int(np.argmax(candidate))]},
)

TUTORIAL_RESULT::{"check": "QAOA cost landscape atol=3e-6", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"best_parameters": [0.5235987755982988, 0.39269908169872414], "max_cost_error": 9.802331946140441e-07, "mettleq_best": 1.9620184302330017, "reference_best": 1.9620190528383281}, "mettleq_median_ms": 139.26999998511747, "notebook": "qiskit/11_qaoa_maxcut.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 13.176207983633503, "reference_over_mettleq": 0.09460909014893032, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
